# Clase 3 · Tokenización — Notebook

Este notebook acompaña al deck de la clase. La idea es **ver la implementación real** de lo que explicamos, lo más simple posible, para fijar los conceptos.

Dos partes:

- **Parte A — Construir un tokenizer mínimo.** De bytes a BPE, con `encode`/`decode`. Todo con funciones sueltas (nada de clases), imprimiendo los resultados intermedios en cada paso.
- **Parte B — Explorar tokenizers reales.** `tiktoken` (GPT) y HuggingFace `AutoTokenizer` para comparar familias y ver "en vivo" las rarezas de la última parte del deck.

> La versión ordenada y con clases de la Parte A es, justamente, el **ejercicio de minbpe** (para casa). Acá lo hacemos a mano para entender el mecanismo.

In [1]:
# Librerías (en Colab se instalan la primera vez)
!pip install -q tiktoken transformers sentencepiece regex

## Parte A · Construir un tokenizer mínimo

Seguimos el mismo camino que el deck: `texto → code points → bytes UTF-8 → BPE → encode/decode`.

### A1 · De texto a bytes

Una computadora no procesa texto, sino números. **Unicode** le asigna a cada carácter un entero (su *code point*); **UTF-8** convierte cada code point en 1 a 4 **bytes** (números de 0 a 255).

In [24]:
# code points: el número que Unicode le da a cada carácter
print("code points:", "a-->", ord("a"), "ñ -->", ord("ñ"), "€--> ", ord("€"))
print("de vuelta:  ", "97-->", chr(97), "241-->", chr(241), "8364-->", chr(8364))
# hoy hay ~150.000 code points asignados en el estándar

code points: a--> 97 ñ --> 241 €-->  8364
de vuelta:   97--> a 241--> ñ 8364--> €


**De code points a bytes (UTF-8).** El code point de la `ñ` es 241, pero se codifica como **dos bytes**: `[195, 177]`. UTF-8 transforma el número, no lo copia.

In [29]:
print("ñ  →", list("ñ".encode("utf-8")))     # [195, 177]
print(30*"-")

s = "canción"
b = list(s.encode("utf-8"))
print(s, "→", b)
print("caracteres:", len(s), "· bytes:", len(b))   # la 'ó' ocupa 2 bytes → más bytes que caracteres
print(30*"-")

s = "hello"
b = list(s.encode("utf-8"))
print(s, "→", b)
print("caracteres:", len(s), "· bytes:", len(b))
print(30*"-")

s = "안녕하세요"
b = list(s.encode("utf-8"))
print(s, "→", b)
print("caracteres:", len(s), "· bytes:", len(b))


ñ  → [195, 177]
------------------------------
canción → [99, 97, 110, 99, 105, 195, 179, 110]
caracteres: 7 · bytes: 8
------------------------------
hello → [104, 101, 108, 108, 111]
caracteres: 5 · bytes: 5
------------------------------
안녕하세요 → [236, 149, 136, 235, 133, 149, 237, 149, 152, 236, 132, 184, 236, 154, 148]
caracteres: 5 · bytes: 15


Cada carácter no-ASCII ocupa 2 a 4 bytes, así que la secuencia de bytes es **más larga** que el texto. Los bytes son un buen punto de partida (alfabeto fijo de 256), pero necesitamos **comprimir** esa secuencia. Eso es BPE.

### A2 · BPE: entrenar

El "entrenamiento" es **puro conteo**: contar los pares de tokens adyacentes, fusionar el más frecuente en un token nuevo, y repetir. Dos funciones alcanzan.

In [30]:
def get_stats(ids):
    # cuenta cuántas veces aparece cada par adyacente
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

ejemplo = list("aaab".encode("utf-8"))
print("ids:  ", ejemplo)
print("pares:", get_stats(ejemplo))

ids:   [97, 97, 97, 98]
pares: {(97, 97): 2, (97, 98): 1}


In [5]:
def merge(ids, pair, idx):
    # reemplaza todas las ocurrencias de 'pair' por el token 'idx'
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

# fusionar el par (97,97) en el token nuevo 256
print(merge([97, 97, 97, 98], (97, 97), 256))   # [256, 97, 98]

[256, 97, 98]


**Sanity check con el ejemplo de la clase: `aaabdaaabac`.** Corremos 3 merges (láminas 14–15).

In [6]:
text = "aaabdaaabac"
ids = list(text.encode("utf-8"))
print("bytes:", ids)

merges = {}
work = list(ids)
for i in range(3):
    stats = get_stats(work)
    pair = max(stats, key=stats.get)   # el par más frecuente
    idx = 256 + i
    print(f"  merge {pair} -> {idx}")
    work = merge(work, pair, idx)
    merges[pair] = idx

print("final:", work)      # [258, 100, 258, 97, 99]
print("merges:", merges)

bytes: [97, 97, 97, 98, 100, 97, 97, 97, 98, 97, 99]
  merge (97, 97) -> 256
  merge (256, 97) -> 257
  merge (257, 98) -> 258
final: [258, 100, 258, 97, 99]
merges: {(97, 97): 256, (256, 97): 257, (257, 98): 258}


El resultado final coincide con la lámina 14: `[258, 100, 258, 97, 99]`.

> **Nota sobre empates.** En la lámina, el 2.º merge era `ab`. Acá el código rompe un empate (dos pares distintos aparecen 2 veces) y elige otro, así que el token intermedio 257 difiere. **El resultado final es el mismo** — los empates se pueden romper de varias formas. Es un detalle de implementación, no del algoritmo.

**Ahora sobre un texto real**, con un vocabulario un poco más grande.

In [7]:
texto = ("La tokenizacion convierte texto en una secuencia de enteros. "
         "Byte-Pair Encoding parte de los 256 bytes y fusiona los pares "
         "mas frecuentes, una y otra vez, para acortar la secuencia. ") * 4

ids = list(texto.encode("utf-8"))
vocab_size = 276          # 256 bytes + 20 merges
num_merges = vocab_size - 256

merges = {}
work = list(ids)
for i in range(num_merges):
    stats = get_stats(work)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    work = merge(work, pair, idx)
    merges[pair] = idx

print("largo original (bytes):", len(ids))
print("largo tokenizado:      ", len(work))
print(f"ratio de compresion:   {len(ids) / len(work):.2f}x")

largo original (bytes): 728
largo tokenizado:       464
ratio de compresion:   1.57x


**Las dos tablas que deja el entrenamiento: `merges` y `vocab`.** `merges` mapea par → id nuevo (lo usa `encode`); `vocab` mapea id → bytes (lo usa `decode`).

In [8]:
# vocab: id -> bytes. Los primeros 256 son los bytes; el resto se arma con los merges
vocab = {i: bytes([i]) for i in range(256)}
for (a, b), idx in merges.items():
    vocab[idx] = vocab[a] + vocab[b]

# ver algunos subwords que aprendio (fijate que capturan espacios + letras frecuentes)
for idx in list(merges.values())[:12]:
    print(idx, "->", vocab[idx])

256 -> b'a '
257 -> b'te'
258 -> b'en'
259 -> b's '
260 -> b'ar'
261 -> b'ci'
262 -> b'on'
263 -> b'ec'
264 -> b'ecu'
265 -> b'ecuen'
266 -> b'par'
267 -> b'to'


El tamaño del vocabulario (`vocab_size`) es un **hiperparámetro**: más grande = secuencias más cortas, pero matrices más grandes en el modelo.

### A3 · encode / decode

Con el reglamento (`merges` + `vocab`) ya podemos traducir en las dos direcciones. `decode` es lo más simple: un lookup.

In [39]:
def decode(ids):
    raw = b"".join(vocab[i] for i in ids)
    return raw.decode("utf-8", errors="replace")

# token por token: cada id y a qué texto corresponde
for w in work[:20]:
    print(f"{w} --> {decode([w])}")

print()
print(decode(work[:20]))

76 --> L
256 --> a 
267 --> to
107 --> k
258 --> en
105 --> i
122 --> z
97 --> a
261 --> ci
262 --> on
32 -->  
99 --> c
262 --> on
118 --> v
105 --> i
101 --> e
114 --> r
268 --> te 
257 --> te
120 --> x

La tokenizacion convierte tex


**`encode`** aplica los merges en el **orden en que se aprendieron** (prioridad de merge), no por frecuencia. Por eso elegimos el par con el id de merge más chico.

In [40]:
def encode(text):
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        # el par cuyo merge se aprendio primero (menor id); inf si no esta en merges
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break        # ya no queda nada por fusionar
        tokens = merge(tokens, pair, merges[pair])
    return tokens

work = encode("Byte-Pair Encoding")
print(work)

[66, 121, 257, 45, 80, 97, 105, 114, 32, 69, 110, 99, 111, 100, 105, 110, 103]


In [41]:
for w in work:
    print(f"{w} --> {decode([w])}")

66 --> B
121 --> y
257 --> te
45 --> -
80 --> P
97 --> a
105 --> i
114 --> r
32 -->  
69 --> E
110 --> n
99 --> c
111 --> o
100 --> d
105 --> i
110 --> n
103 --> g


**Round-trip y compresión: el checkpoint** (lámina 19). `decode(encode(texto)) == texto` para todo texto, y medimos cuánto acortamos.

In [11]:
s = "de bytes a tokens, y de vuelta."
print("round-trip exacto:      ", decode(encode(s)) == s)
print("round-trip sobre corpus:", decode(encode(texto)) == texto)
print(f"compresion: {len(texto.encode('utf-8'))} bytes -> {len(encode(texto))} tokens")

round-trip exacto:       True
round-trip sobre corpus: True
compresion: 728 bytes -> 464 tokens


Reversible y comprime. Con esto tenemos un tokenizer completo a nivel *BasicTokenizer*.

> El `RegexTokenizer` completo (entrenar por fragmento) y **reproducir GPT-4** son los pasos del **ejercicio de minbpe**.

### A4 · Regex splitting (GPT-2 / GPT-4)

Sin restricciones, BPE fusiona a través de los límites de palabra (`perro.`, `perro!`… como tokens distintos). GPT antepone un **patrón (regex)** que corta el texto por categorías —letras, números, puntuación, espacios— y BPE corre por separado dentro de cada fragmento.

In [12]:
import regex as re

# patron simplificado de GPT-2 (usa la libreria regex, no re, por \p{L} y \p{N})
patron = re.compile(r"'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+")

print(re.findall(patron, "Hola mundo 123!"))
print(re.findall(patron, "for i in range(10): print(i)"))

['Hola', ' mundo', ' 123', '!']
['for', ' i', ' in', ' range', '(', '10', '):', ' print', '(', 'i', ')']


Cada categoría queda por separado y el espacio se pega a la palabra siguiente (`" mundo"`). Así BPE nunca fusiona una letra con un signo o con un número.

## Parte B · Explorar tokenizers reales

Hasta acá construimos **un** tokenizer (BPE sobre bytes, la familia GPT). Ahora usamos los reales para ver los distintos algoritmos y las rarezas del deck.

### B1 · tiktoken (la librería de GPT)

`tiktoken` implementa el BPE byte-level de OpenAI. **Solo infiere**: trae los vocabularios ya entrenados (`r50k`, `cl100k`, `o200k`) y hace `encode`/`decode`.

In [44]:
import tiktoken

def mostrar(enc, nombre, texto):
    ids = enc.encode(texto)
    print(f"--- {nombre}  (vocab: {enc.n_vocab}) ---")
    print("ids:", ids)
    for i in ids:
        print(f"  {i:>6} --> {enc.decode_single_token_bytes(i)}")
    print("texto:", enc.decode(ids))
    print()

cl100k = tiktoken.get_encoding("cl100k_base")   # GPT-3.5 / GPT-4
o200k  = tiktoken.get_encoding("o200k_base")    # GPT-4o

mostrar(cl100k, "cl100k", "La tokenizacion en accion")
mostrar(o200k,  "o200k",  "La tokenizacion en accion")

--- cl100k  (vocab: 100277) ---
ids: [8921, 4037, 52570, 665, 1046, 290]
    8921 --> b'La'
    4037 --> b' token'
   52570 --> b'izacion'
     665 --> b' en'
    1046 --> b' acc'
     290 --> b'ion'
texto: La tokenizacion en accion

--- o200k  (vocab: 200019) ---
ids: [4579, 6602, 67707, 469, 119285]
    4579 --> b'La'
    6602 --> b' token'
   67707 --> b'izacion'
     469 --> b' en'
  119285 --> b' accion'
texto: La tokenizacion en accion



**GPT-2 vs GPT-4 con los espacios.** GPT-4 fusiona corridas de espacios; GPT-2 no. Es un cambio del patrón de regex.

In [45]:
gpt2 = tiktoken.get_encoding("gpt2")
print("GPT-2:", gpt2.encode("    hello world"))
print("GPT-4:", cl100k.encode("    hello world"))

GPT-2: [220, 220, 220, 23748, 995]
GPT-4: [262, 24748, 1917]


**Tokens especiales.** Hay que habilitarlos explícitamente (`allowed_special`); si no, `tiktoken` tira error a propósito (interpretar cadenas de control en input de usuario es un riesgo).

In [49]:
try:
  print(cl100k.encode("<|endoftext|>hola"))
except ValueError as e:
  print(f"ERROR:\n\n{e}")

ERROR:

Encountered text corresponding to disallowed special token '<|endoftext|>'.
If you want this text to be encoded as a special token, pass it to `allowed_special`, e.g. `allowed_special={'<|endoftext|>', ...}`.
If you want this text to be encoded as normal text, disable the check for this token by passing `disallowed_special=(enc.special_tokens_set - {'<|endoftext|>'})`.
To disable this check for all special tokens, pass `disallowed_special=()`.



In [50]:
print(cl100k.encode("<|endoftext|>hola", allowed_special="all"))

[100257, 71, 8083]


### B2 · Comparar algoritmos con `AutoTokenizer`

El mismo texto por tres familias: **GPT-2** (BPE), **BERT** (WordPiece → prefijo `##`) y **T5** (SentencePiece + Unigram → símbolo `▁` para el espacio).

In [16]:
from transformers import AutoTokenizer

texto_demo = "unbelievable tokenization"
for nombre in ["gpt2", "bert-base-uncased", "t5-small"]:
    tok = AutoTokenizer.from_pretrained(nombre)
    print(f"{nombre:18s} {tok.tokenize(texto_demo)}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

t5-small           ['▁unbelievable', '▁token', 'ization']


Fijate: BERT parte `unbelievable` y marca la continuación con `##`; T5 usa `▁` para señalar dónde había un espacio. Son los algoritmos de las láminas 36–39, en la naturaleza.

### B3 · Las rarezas, en vivo

Cada comportamiento raro de la lámina 40 se ve concretamente mirando cómo se parte el texto.

In [17]:
# DELETREO: el modelo ve tokens, no letras -> no puede contar las "r"
gpt2 = AutoTokenizer.from_pretrained("gpt2")
print("strawberry →", gpt2.tokenize("strawberry"))

strawberry → ['st', 'raw', 'berry']


In [18]:
# ARITMETICA: los numeros se parten de formas inconsistentes
for n in ["7", "123", "1234", "12345"]:
    print(f"{n:6s} → {cl100k.encode(n)}")

7      → [22]
123    → [4513]
1234   → [4513, 19]
12345  → [4513, 1774]


In [19]:
# OTROS IDIOMAS: mas tokens por palabra fuera del ingles
for s in ["I love tokenizers", "Amo los tokenizers", "\u30c8\u30fc\u30af\u30ca\u30a4\u30b6\u30fc\u304c\u597d\u304d"]:
    print(f"{len(cl100k.encode(s)):3d} tokens · {len(s):3d} chars · {s}")

  4 tokens ·  17 chars · I love tokenizers
  5 tokens ·  18 chars · Amo los tokenizers
 11 tokens ·  10 chars · トークナイザーが好き


In [20]:
# ESPACIOS: un espacio final cambia la tokenizacion
print("'hola'  →", cl100k.encode("hola"))
print("'hola ' →", cl100k.encode("hola "))

'hola'  → [71, 8083]
'hola ' → [71, 8083, 220]


In [21]:
# FORMATO: el mismo dato cuesta distinta cantidad de tokens
json_s = '{"nombre": "Ana", "edad": 30}'
yaml_s = "nombre: Ana\nedad: 30"
print("JSON:", len(cl100k.encode(json_s)), "tokens")
print("YAML:", len(cl100k.encode(yaml_s)), "tokens")

JSON: 12 tokens
YAML: 8 tokens


Ninguna es magia: todas se siguen de que el texto se representa como **tokens**, no como caracteres — exactamente lo que fuimos construyendo.

## Parte C · Cierre

Construimos un tokenizer mínimo desde cero y vimos los reales por dentro.

- **Opcional:** el **ejercicio de minbpe** — armá tu propio tokenizer de GPT-4, con `RegexTokenizer` y reproduciendo `cl100k` vía `tiktoken`.
- **Lo que sigue en el curso:** cada token entra al modelo como un **vector** (embedding). Cómo se construye ese `token → vector` es el próximo tema.